<a href="https://colab.research.google.com/github/gunnsmart/science-skills/blob/arena%2F01a0bfbe-science-skills/Image_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Generator — Open-model Workbench

**เลือกโมเดล → ตั้งค่า → สร้างภาพ → ดาวน์โหลด PNG / JPEG + metadata**

Notebook แบบ standalone สำหรับ **Google Colab / Jupyter** ไม่ต้อง clone repository และไม่มี web server

## ใช้งานแบบง่าย — ไม่ต้องแก้โค้ด
1. Colab: เลือก **Runtime → Change runtime type → GPU**
2. กด **Runtime → Run all** ครั้งแรก เพื่อเตรียมระบบ (ยังไม่โหลด weights / ยังไม่สร้างภาพ)
3. เลื่อนไปที่แผง **🎨 Image Generator** ด้านล่างสุดของ Notebook
4. **เลือกโมเดล → ใส่ Prompt → เลือกสัดส่วน → กด “สร้างภาพ”** แล้วดาวน์โหลด ZIP จากแผงเดียวกัน

มีสัดส่วน **1:1, 3:2, 2:3, 4:3, 3:4, 16:9, 9:16** และขนาดกำหนดเอง พร้อมแสดง pixel จริง
Steps / Guidance เติมค่าที่แนะนำเมื่อเลือกโมเดล; Negative prompt, Seed, LoRA และการจัดการหน่วยความจำอยู่ในเมนูพับ “ขั้นสูง”

- เริ่มด้วย **SD 1.5** บน GPU เล็ก หรือ SDXL บน runtime ที่เหมาะสม; FLUX / Qwen ใช้ RAM / VRAM สูงและอาจไม่พอดีกับ Colab ฟรี
- เปลี่ยนโมเดลแล้ว Prompt / สัดส่วน / Seed ยังอยู่ แต่จะคืนค่า Steps / Guidance และล้าง LoRA / Extra kwargs เพื่อไม่ให้ค่าของโมเดลเก่าปนกัน
- **ไม่ต้อง Run all ซ้ำเมื่อปรับค่า** กดสร้างภาพจากแผงได้เลย; ถ้า restart runtime ให้ Run all ใหม่
- ถ้าเคย import dependencies เวอร์ชันเก่าก่อนติดตั้ง ให้ restart runtime หลังติดตั้งก่อนใช้งาน
- ถ้าแผงไม่แสดง / ปุ่มไม่ตอบสนอง ให้รันเซลล์แผงใหม่ และตรวจว่า runtime ยังเชื่อมต่อ; Jupyter ต้องเปิด widget support
- การหยุดงานใช้ **Runtime → Interrupt execution** หรือปุ่ม Stop ของ Notebook ไม่ใช่ปุ่มในหน้าเว็บแยก

### ขอบเขตที่รองรับจริง
- Text-to-image ผ่าน **Diffusers pipelines ที่ติดตั้งอยู่** ไม่จำกัดเฉพาะรายชื่อ preset: repo ใหม่ / fine-tune ใช้ Custom ได้เมื่อสถาปัตยกรรมรองรับ
- Single-file checkpoint ต้องมีตัวแปลง `from_single_file` และเลือก pipeline ให้ตรงสถาปัตยกรรม; `.safetensors` **ไม่ใช่**รูปแบบที่ทำให้ทุกโมเดลโหลดร่วมกันได้
- โมเดลที่ใช้โค้ดเฉพาะ, GGUF, quantization เฉพาะทาง, multi-stage workflow, image-editing, ControlNet และ inpainting ไม่ได้รองรับโดยอัตโนมัติ ต้องเขียน local adapter (ตัวอย่างท้าย notebook)
- **ไม่รับประกัน “ทุกโมเดล Open source”**: open weights ไม่เท่ากับ open-source license หรืออนุญาตเชิงพาณิชย์ อ่าน model card/license ของแต่ละ repo ก่อนใช้; preset เป็นค่าเริ่มต้น ไม่ใช่ผลทดสอบ GPU ทุกตัว
- ไม่เปิด remote Python code และไม่โหลด pickle `.ckpt` โดยอัตโนมัติ ไม่ปิด safety checker ที่โมเดลมีให้

ภาพ, prompt, seed และ config จะอยู่ใน ZIP: อย่าแชร์ metadata หาก prompt มีข้อมูลส่วนตัว ระบบปิดบัง token ที่รู้จักใน metadata/ข้อความ error ของแผง แต่ยังต้องตรวจไฟล์และ output ก่อนแชร์

In [ ]:
#@title 1 — Install dependencies (รันก่อน import)
INSTALL_DEPENDENCIES = True #@param {type:"boolean"}
import importlib.util
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    # Keep Colab's CUDA-compatible torch rather than force-replacing it.
    if importlib.util.find_spec("torch") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "torch", "torchvision"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade",
        "diffusers>=0.36.0,<1", "transformers>=4.51,<5", "accelerate>=1.2,<2",
        "huggingface_hub>=0.34,<1", "safetensors>=0.4", "sentencepiece",
        "protobuf", "pillow>=10", "peft>=0.15,<1", "ipywidgets>=8,<9"], check=True)
    print("ติดตั้งแล้ว — ถ้าเคย import dependencies ใน runtime นี้ ให้ restart ก่อนรันเซลล์ถัดไป")

In [ ]:
#@title 2 — Environment + optional Hugging Face authentication
import gc
import os
import platform
from importlib.metadata import version
from pathlib import Path
import torch
import diffusers
from PIL import Image
from IPython.display import display, FileLink

WORK_DIR = Path("/content/ImageGenerator" if Path("/content").exists() else "image_generator_outputs").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
HAS_CUDA = torch.cuda.is_available()
print("Python:", platform.python_version())
print({name: version(name) for name in ("torch", "diffusers", "transformers", "accelerate")})
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "| VRAM:", round(props.total_memory / 1024**3, 1), "GiB")
else:
    print("CPU mode: ช้ามากและโมเดลใหญ่ใช้ RAM สูง — แนะนำ GPU runtime")

# Optional: Colab Secrets named HF_TOKEN, or environment HF_TOKEN, or existing HF login.
# Never paste a token into source code / form fields or include it in saved metadata.
HF_TOKEN = os.environ.get("HF_TOKEN") or None
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF authentication:", "secret supplied" if HF_TOKEN else "cached login / public access")
print("Gated repos: ยอมรับ license และขอสิทธิ์ใน model card ด้วยบัญชีเจ้าของ token ก่อน")

In [ ]:
#@title 3 — Model registry (เพิ่ม preset ได้ ไม่ได้จำกัด Custom models)
# class names are built-in Diffusers classes, NOT downloaded Python modules.
# RAM/VRAM depends on dtype, resolution, offload, runtime and model revision.
MODEL_REGISTRY = {
    "SD 1.5": dict(repo="stable-diffusion-v1-5/stable-diffusion-v1-5", pipeline="StableDiffusionPipeline", size=512, steps=30, guidance=7.5, multiple=8, precision="FP16", note="OpenRAIL; จุดเริ่มต้นสำหรับ GPU เล็ก"),
    "SDXL": dict(repo="stabilityai/stable-diffusion-xl-base-1.0", pipeline="StableDiffusionXLPipeline", size=1024, steps=30, guidance=5.0, multiple=8, precision="FP16", note="OpenRAIL++; base pipeline ไม่รวม refiner"),
    "SDXL Turbo": dict(repo="stabilityai/sdxl-turbo", pipeline="StableDiffusionXLPipeline", size=512, steps=4, guidance=0.0, multiple=8, precision="FP16", note="ตรวจ license; distilled model, CFG=0"),
    "SD 3.5 Medium": dict(repo="stabilityai/stable-diffusion-3.5-medium", pipeline="StableDiffusion3Pipeline", size=1024, steps=28, guidance=4.5, multiple=16, precision="BF16", note="Gated / Stability Community License; RAM สูง"),
    "FLUX.1 schnell": dict(repo="black-forest-labs/FLUX.1-schnell", pipeline="FluxPipeline", size=1024, steps=4, guidance=0.0, multiple=16, precision="BF16", extra={"max_sequence_length":256}, note="Apache-2.0; โมเดลใหญ่; ไม่ใช้ negative prompt แบบ SD"),
    "FLUX.1 dev": dict(repo="black-forest-labs/FLUX.1-dev", pipeline="FluxPipeline", size=1024, steps=28, guidance=3.5, multiple=16, precision="BF16", note="Gated / non-commercial model license; ไม่ใช่ unrestricted open source"),
    "PixArt Sigma": dict(repo="PixArt-alpha/PixArt-Sigma-XL-2-1024-MS", pipeline="PixArtSigmaPipeline", size=1024, steps=20, guidance=4.5, multiple=16, precision="FP16", note="ตรวจ model card/license; T5 text encoder ใช้ RAM เพิ่ม"),
    "Sana": dict(repo="Efficient-Large-Model/Sana_1600M_1024px_diffusers", pipeline="SanaPipeline", size=1024, steps=20, guidance=5.0, multiple=32, precision="BF16", note="ตรวจ model card/license; อาจต้องสิทธิ์ text encoder"),
    "Lumina 2": dict(repo="Alpha-VLLM/Lumina-Image-2.0", pipeline="Lumina2Pipeline", size=1024, steps=30, guidance=4.0, multiple=32, precision="BF16", note="ตรวจ model card/license; ใช้ BF16 หรือ FP32"),
    "AuraFlow": dict(repo="fal/AuraFlow-v0.3", pipeline="AuraFlowPipeline", size=1024, steps=50, guidance=3.5, multiple=16, precision="FP16", note="ตรวจ model card/license; โมเดลใหญ่"),
    "HunyuanDiT": dict(repo="Tencent-Hunyuan/HunyuanDiT-Diffusers", pipeline="HunyuanDiTPipeline", size=1024, steps=50, guidance=5.0, multiple=16, precision="FP16", note="ตรวจ Tencent license; text-to-image ไม่ใช่ video"),
    "Qwen Image": dict(repo="Qwen/Qwen-Image", pipeline="QwenImagePipeline", size=1024, steps=50, guidance=4.0, guidance_parameter="true_cfg_scale", multiple=16, precision="BF16", note="Apache-2.0; โมเดลใหญ่มาก ไม่เหมาะกับ Colab RAM ต่ำ"),
    "Z-Image Turbo": dict(repo="Tongyi-MAI/Z-Image-Turbo", pipeline="ZImagePipeline", size=1024, steps=9, guidance=0.0, multiple=16, precision="BF16", note="Apache-2.0; ต้องใช้ Diffusers ที่มี ZImagePipeline"),
}

for name, preset in MODEL_REGISTRY.items():
    # Inspect the lazy module namespace without importing every optional pipeline.
    available = preset["pipeline"] in dir(diffusers)
    print(f"{name}: {'class installed' if available else 'upgrade Diffusers required'} | {preset['note']}")
    print("  Model card:", "https://huggingface.co/" + preset["repo"])
print("class installed ≠ weights downloaded / authenticated / tested on this GPU")

In [ ]:
#@title ⚙️ Internal defaults — ใช้แผงควบคุมด้านล่างแทนการแก้เซลล์นี้
MODEL = "SD 1.5"
CUSTOM_MODEL = ""
SOURCE = "Repository / directory"
PIPELINE_CLASS = "Auto"
# Auto uses the preset class, or model_index.json for Custom repositories.
# Custom single file: enter e.g. StableDiffusionXLPipeline explicitly.
SINGLE_FILE_CONFIG = ""
# Optional matching Diffusers config repo/directory for from_single_file.
REVISION = ""
# Pin a commit SHA for a HF repo (blank = default branch / revision in single-file URL).
# For single-file checkpoints this pins weights, not the separate config repo.
BACKEND = "diffusers"
PROMPT = "A tiny observatory on a green mountain, sunrise, soft clouds, detailed landscape photography"
NEGATIVE_PROMPT = ""
WIDTH = 0
HEIGHT = 0
STEPS = 0
GUIDANCE = -1.0
SEED = 42
# -1 = random base seed; each image gets base_seed + index.
NUM_IMAGES = 1
PRECISION = "Auto"
MEMORY_MODE = "Auto"
VAE_TILING = True
LORA_SOURCE = ""
LORA_WEIGHT_NAME = ""
LORA_SCALE = 1.0
# Optional local/HF LoRA; must match the model architecture. Leave blank to disable.
EXTRA_KWARGS_JSON = "{}"
# Example for supported pipelines: {"max_sequence_length": 256}. Qwen CFG uses GUIDANCE above.
# Unknown keys and overrides of the active guidance control fail, never silently ignored.
EXPORT_JPEG = True
KEEP_MODEL_IN_MEMORY = False
# Default unloads after generation to free RAM/VRAM; weights stay in HF disk cache.

# Scalar configuration only: never keep runtime objects, model references or tokens in the UI.
SETTING_FIELDS = ('MODEL', 'CUSTOM_MODEL', 'SOURCE', 'PIPELINE_CLASS', 'SINGLE_FILE_CONFIG', 'REVISION', 'BACKEND', 'PROMPT', 'NEGATIVE_PROMPT', 'WIDTH', 'HEIGHT', 'STEPS', 'GUIDANCE', 'SEED', 'NUM_IMAGES', 'PRECISION', 'MEMORY_MODE', 'VAE_TILING', 'LORA_SOURCE', 'LORA_WEIGHT_NAME', 'LORA_SCALE', 'EXTRA_KWARGS_JSON', 'EXPORT_JPEG', 'KEEP_MODEL_IN_MEMORY')
DEFAULT_SETTINGS = {name: globals()[name] for name in SETTING_FIELDS}


In [ ]:
#@title 5 — Validation / pipeline argument routing (ไม่มีการโหลด weights)
import inspect
import json
import math
import secrets
import re
from urllib.parse import urlparse, unquote, quote, quote_plus


def safe_relative_filename(value, label="filename"):
    # Check the decoded string; checking the URL before unquote misses %2F / %2E%2E.
    if not isinstance(value, str) or not value or "\\" in value or ":" in value:
        raise ValueError(f"{label} ต้องเป็น relative path ภายใน repository")
    if any(ord(char) < 32 or ord(char) == 127 for char in value):
        raise ValueError(f"{label} มี control characters")
    if any(part in ("", ".", "..") for part in value.split("/")):
        raise ValueError(f"{label} ห้ามมี absolute path, . หรือ ..")
    return value


def validate_repo_or_path(value, label):
    parsed = urlparse(value)
    if parsed.scheme or parsed.netloc or parsed.query or parsed.fragment:
        raise ValueError(f"{label}: ใช้ repo ID หรือ local path ไม่ใช่ URL; เก็บ token ใน Colab Secrets")
    if any(ord(char) < 32 or ord(char) == 127 for char in value):
        raise ValueError(f"{label}: ไม่อนุญาต control characters")


def redact_text(value):
    """Defense in depth for notebook-owned messages; not a third-party log filter."""
    text = str(value)
    token = globals().get("HF_TOKEN")
    if isinstance(token, str) and token:
        for spelling in {token, quote(token, safe=""), quote_plus(token)}:
            text = text.replace(spelling, "[REDACTED]")
    text = re.sub(r"\bhf_[A-Za-z0-9]{16,}\b", "[REDACTED]", text)
    text = re.sub(r"(?i)(https?://)[^/\s@]+@", r"\1[REDACTED]@", text)
    text = re.sub(r"(?i)([?&](?:token|access_token|api_key|apikey|authorization|password|secret)=)[^&#\s]+",
                  r"\1[REDACTED]", text)
    return text


def redact_metadata(value):
    if isinstance(value, dict):
        secret_keys = {"token", "hf_token", "access_token", "api_key", "apikey", "authorization", "password", "secret"}
        return {redact_text(key): "[REDACTED]" if str(key).lower() in secret_keys else redact_metadata(item)
                for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [redact_metadata(item) for item in value]
    return redact_text(value) if isinstance(value, str) else value


def parse_extra_kwargs(raw):
    if not isinstance(raw, str) or len(raw.encode("utf-8")) > 32768:
        raise ValueError("Extra kwargs ต้องเป็น JSON ไม่เกิน 32 KiB")
    try:
        extras = json.loads(raw, parse_constant=lambda x: (_ for _ in ()).throw(ValueError("Non-finite JSON")))
    except (RecursionError, json.JSONDecodeError) as exc:
        raise ValueError("Extra kwargs เป็น JSON ไม่ถูกต้องหรือซ้อนลึกเกินไป") from None
    if not isinstance(extras, dict):
        raise ValueError("EXTRA_KWARGS_JSON ต้องเป็น JSON object")
    def check_depth(value, depth=0):
        if depth > 12:
            raise ValueError("Extra kwargs ซ้อนลึกเกิน 12 ชั้น")
        if isinstance(value, dict):
            for item in value.values():
                check_depth(item, depth + 1)
        elif isinstance(value, list):
            for item in value:
                check_depth(item, depth + 1)
    check_depth(extras)
    json.dumps(extras, allow_nan=False)
    return extras


def single_file_location(source):
    """Parse only supported HF URLs; other hosts must be downloaded locally first."""
    if any(ord(char) < 32 or ord(char) == 127 for char in source):
        raise ValueError("Checkpoint source มี control characters")
    parsed = urlparse(source)
    if not parsed.scheme and not parsed.netloc:
        # urlparse strips query/fragment; a literal local filename does not.
        # Reject the ambiguous spelling rather than validate one suffix and load another.
        if "?" in source or "#" in source:
            raise ValueError("Local checkpoint path ต้องไม่มี query หรือ fragment")
        return None
    if parsed.scheme != "https" or parsed.netloc not in ("huggingface.co", "hf.co"):
        raise ValueError("Single-file URL ต้องเป็น HTTPS ของ huggingface.co/hf.co; แหล่งอื่นให้ดาวน์โหลดเป็น local file ก่อน")
    if parsed.query or parsed.fragment:
        raise ValueError("ใช้ URL ที่ไม่มี query/fragment/token; ยืนยันตัวตนผ่าน HF_TOKEN เท่านั้น")
    parts = parsed.path.strip("/").split("/", 4)
    if len(parts) != 5 or parts[2] not in ("resolve", "blob") or not all(parts):
        raise ValueError("ใช้ URL รูปแบบ https://huggingface.co/org/repo/resolve/revision/file.safetensors")
    namespace, repository, revision, filename = (unquote(parts[i]) for i in (0, 1, 3, 4))
    for slug in (namespace, repository):
        if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_.-]*", slug):
            raise ValueError("Invalid Hugging Face repository ID")
    safe_relative_filename(revision, "revision")
    safe_relative_filename(filename, "checkpoint filename")
    if not filename.endswith(".safetensors"):
        raise ValueError("Checkpoint filename ต้องลงท้าย .safetensors ตัวพิมพ์เล็ก")
    return dict(repo_id=namespace + "/" + repository, revision=revision, filename=filename)


def resolve_settings(values):
    model = values["MODEL"]
    if model != "Custom" and model not in MODEL_REGISTRY:
        raise ValueError("Unknown preset: " + model)
    preset = MODEL_REGISTRY.get(model, {})
    source = values["CUSTOM_MODEL"].strip() if model == "Custom" else preset["repo"]
    if not source:
        raise ValueError("Custom: ใส่ Hugging Face model ID หรือ local model path")
    pipeline = values["PIPELINE_CLASS"].strip()
    if pipeline == "Auto":
        pipeline = preset.get("pipeline", "Auto")
    if values["SOURCE"] not in ("Repository / directory", "Single safetensors file"):
        raise ValueError("Unknown SOURCE")
    if values["SOURCE"] == "Single safetensors file":
        if model != "Custom":
            raise ValueError("Single file: เลือก Custom แล้วใส่ path/HTTPS URL ของไฟล์")
        parsed = urlparse(source)
        if not parsed.path.endswith(".safetensors"):
            raise ValueError("รับเฉพาะ .safetensors ไม่โหลด pickle .ckpt/.bin")
        location = single_file_location(source)
        revision = values["REVISION"].strip()
        if location and revision and revision != location["revision"]:
            raise ValueError("REVISION ไม่ตรงกับ revision ใน URL; แก้ให้ตรงกันหรือเว้น REVISION ว่าง")
        if pipeline == "Auto":
            raise ValueError("Single file ต้องระบุ PIPELINE_CLASS ให้ตรงสถาปัตยกรรม")
    else:
        validate_repo_or_path(source, "Model source")
    for field in ("LORA_SOURCE", "SINGLE_FILE_CONFIG"):
        validate_repo_or_path(values[field].strip(), field)
    if values["LORA_WEIGHT_NAME"].strip():
        filename = safe_relative_filename(values["LORA_WEIGHT_NAME"].strip(), "LoRA filename")
        if not filename.endswith(".safetensors"):
            raise ValueError("LoRA filename ต้องลงท้าย .safetensors ตัวพิมพ์เล็ก")
    if not isinstance(values["PROMPT"], str) or not values["PROMPT"].strip():
        raise ValueError("Prompt ต้องไม่ว่าง")
    for name in ("WIDTH", "HEIGHT", "STEPS", "SEED", "NUM_IMAGES"):
        if type(values[name]) is not int:
            raise ValueError(name + " ต้องเป็นจำนวนเต็ม")
    # Conservative Custom alignment; architecture-specific restrictions may be stricter.
    multiple = preset.get("multiple", 32)
    width = values["WIDTH"] or preset.get("size", 1024)
    height = values["HEIGHT"] or preset.get("size", 1024)
    if any(n < 128 or n > 4096 or n % multiple for n in (width, height)):
        raise ValueError(f"ขนาดต้องอยู่ในช่วง 128–4096 และหาร {multiple} ลงตัว")
    steps = values["STEPS"] or preset.get("steps", 30)
    if not 1 <= steps <= 200:
        raise ValueError("Steps ต้องอยู่ในช่วง 1–200")
    if not 1 <= values["NUM_IMAGES"] <= 8:
        raise ValueError("NUM_IMAGES ต้องอยู่ในช่วง 1–8")
    seed = values["SEED"]
    if seed != -1 and not 0 <= seed <= 2**32 - values["NUM_IMAGES"]:
        raise ValueError("Seed ต้องเป็น -1 หรืออยู่ในช่วง uint32 ที่เพิ่มตามจำนวนภาพได้")
    if seed == -1:
        seed = secrets.randbelow(2**32 - values["NUM_IMAGES"] + 1)
    guidance = values["GUIDANCE"]
    if guidance == -1:
        guidance = preset.get("guidance", 5.0)
    if not math.isfinite(guidance) or not 0 <= guidance <= 30:
        raise ValueError("Guidance ต้องอยู่ในช่วง 0–30 หรือ -1 เพื่อใช้ preset")
    if not math.isfinite(values["LORA_SCALE"]) or not 0 <= values["LORA_SCALE"] <= 2:
        raise ValueError("LoRA scale ต้องอยู่ในช่วง 0–2")
    if values["PRECISION"] not in ("Auto", "FP16", "BF16", "FP32"):
        raise ValueError("Unknown PRECISION")
    if values["MEMORY_MODE"] not in ("Auto", "GPU", "Model CPU offload", "Sequential CPU offload", "CPU"):
        raise ValueError("Unknown MEMORY_MODE")
    extras = parse_extra_kwargs(values["EXTRA_KWARGS_JSON"])
    reserved = {"prompt", "negative_prompt", "width", "height", "num_inference_steps", "guidance_scale",
                "generator", "num_images_per_prompt", "output_type", "return_dict", "image", "mask_image"}
    if reserved.intersection(extras):
        raise ValueError("Extra kwargs ห้ามทับค่าหลัก: " + str(sorted(reserved.intersection(extras))))
    return dict(model=model, source=source, source_type=values["SOURCE"], pipeline=pipeline,
        revision=values["REVISION"].strip() or None, single_file_config=values["SINGLE_FILE_CONFIG"].strip(),
        backend=values["BACKEND"].strip(), prompt=values["PROMPT"], negative_prompt=values["NEGATIVE_PROMPT"],
        width=width, height=height, steps=steps, guidance=guidance,
        guidance_parameter=preset.get("guidance_parameter", "Auto"), seed=seed, count=values["NUM_IMAGES"],
        precision=values["PRECISION"], preferred_precision=preset.get("precision", "BF16"),
        memory_mode=values["MEMORY_MODE"], vae_tiling=values["VAE_TILING"],
        lora_source=values["LORA_SOURCE"].strip(), lora_weight_name=values["LORA_WEIGHT_NAME"].strip(),
        lora_scale=values["LORA_SCALE"], extra={**preset.get("extra", {}), **extras},
        export_jpeg=values["EXPORT_JPEG"], keep_model=values["KEEP_MODEL_IN_MEMORY"])


def build_call_kwargs(pipe, settings, generator):
    parameters = inspect.signature(pipe.__call__).parameters
    required = {"prompt", "width", "height", "num_inference_steps", "generator"}
    if required.difference(parameters):
        raise ValueError("Pipeline ไม่ตรง text-to-image contract; ต้องใช้ local adapter: " +
                         str(sorted(required.difference(parameters))))
    kwargs = dict(prompt=settings["prompt"], width=settings["width"], height=settings["height"],
                  num_inference_steps=settings["steps"], generator=generator)
    notes = []
    guidance_parameter = settings["guidance_parameter"]
    if guidance_parameter == "Auto":
        # Qwen's base model ignores guidance_scale; CFG is true_cfg_scale.
        # Keep Flux's distilled guidance separate: it also exposes both parameters.
        guidance_parameter = "true_cfg_scale" if type(pipe).__name__ == "QwenImagePipeline" else "guidance_scale"
    if guidance_parameter in parameters:
        kwargs[guidance_parameter] = settings["guidance"]
    else:
        notes.append(f"Pipeline ไม่มี {guidance_parameter}; ไม่ส่งค่านี้")
    if "negative_prompt" in parameters:
        # Empty string is intentional: some true-CFG models need an unconditional prompt.
        kwargs["negative_prompt"] = settings["negative_prompt"]
    elif settings["negative_prompt"]:
        notes.append("Pipeline ไม่มี negative_prompt; ไม่ส่งค่านี้")
    for key, value in settings["extra"].items():
        if key == guidance_parameter:
            raise ValueError(f"ใช้ช่อง GUIDANCE แทน extra kwarg {key} เพื่อไม่ให้ค่าหลักถูกทับ")
        if key not in parameters:
            raise ValueError(f"Pipeline ไม่รองรับ extra kwarg: {key}")
        kwargs[key] = value
    # One image per call avoids a multi-image VRAM spike.
    if "num_images_per_prompt" in parameters:
        kwargs["num_images_per_prompt"] = 1
    if "output_type" in parameters:
        kwargs["output_type"] = "pil"
    if "return_dict" in parameters:
        kwargs["return_dict"] = True
    return kwargs, notes

In [ ]:
#@title 6 — Model loader + memory management + optional LoRA
# Extension point: audited local Python only. See adapter instructions below.
from pathlib import Path
CUSTOM_LOADERS = globals().get("CUSTOM_LOADERS", {})
PIPE = globals().get("PIPE", None)
PIPE_KEY = globals().get("PIPE_KEY", None)
PIPE_LOAD_INFO = globals().get("PIPE_LOAD_INFO", {})


def unload_model():
    global PIPE, PIPE_KEY, PIPE_LOAD_INFO
    PIPE = None
    PIPE_KEY = None
    PIPE_LOAD_INFO = {}
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def select_runtime(settings):
    mode = settings["memory_mode"]
    cuda = torch.cuda.is_available()
    if mode == "Auto":
        mode = "Model CPU offload" if cuda else "CPU"
    if mode != "CPU" and not cuda:
        raise ValueError("โหมดนี้ต้องมี CUDA GPU; เลือก GPU runtime หรือ CPU")
    precision = settings["precision"]
    bf16 = cuda and torch.cuda.is_bf16_supported()
    if precision == "Auto":
        precision = "FP32" if mode == "CPU" else settings["preferred_precision"]
        if precision == "BF16" and not bf16:
            precision = "FP32"
            print("GPU ไม่รองรับ BF16: ใช้ FP32 เพื่อเลี่ยง FP16 overflow; RAM/VRAM จะสูงขึ้น")
    if mode == "CPU" and precision != "FP32":
        raise ValueError("CPU mode ใน notebook นี้ใช้ FP32; เลือก Auto หรือ FP32")
    if precision == "BF16" and not bf16:
        raise ValueError("GPU ไม่รองรับ BF16; ใช้ Auto/FP32 หรือ GPU รุ่นที่รองรับ")
    return mode, {"FP16":torch.float16, "BF16":torch.bfloat16, "FP32":torch.float32}[precision]


def load_diffusers(settings, dtype):
    name = settings["pipeline"]
    cls = diffusers.DiffusionPipeline if name == "Auto" else getattr(diffusers, name, None)
    if not isinstance(cls, type) or not issubclass(cls, diffusers.DiffusionPipeline):
        raise ValueError(f"ไม่มี built-in Diffusers pipeline {name}; ตรวจชื่อ/อัปเดต Diffusers แล้ว restart runtime")
    options = dict(torch_dtype=dtype, token=HF_TOKEN)
    if settings["source_type"] == "Single safetensors file":
        if not hasattr(cls, "from_single_file"):
            raise ValueError(name + " ไม่มี from_single_file; ใช้ Diffusers repo/directory หรือ local adapter")
        if settings["single_file_config"]:
            options["config"] = settings["single_file_config"]
        source = settings["source"]
        location = single_file_location(source)
        if location:
            # Diffusers 0.36's own URL parser does not reliably handle resolve/ref URLs.
            # Download explicitly so nested paths and non-main revisions are honored.
            from huggingface_hub import hf_hub_download
            source = hf_hub_download(**location, token=HF_TOKEN)
        return cls.from_single_file(source, **options)
    return cls.from_pretrained(settings["source"], revision=settings["revision"],
                               use_safetensors=True, trust_remote_code=False, **options)


def resolve_lora_source(settings):
    """Resolve one explicit safe filename before base weights load; never guess ambiguously."""
    source, weight_name = settings["lora_source"], settings["lora_weight_name"]
    if not source:
        return None, None
    path = Path(source).expanduser()
    if path.is_file():
        if path.suffix != ".safetensors":
            raise ValueError("Local LoRA ต้องเป็นไฟล์ .safetensors ตัวพิมพ์เล็ก")
        if weight_name and weight_name != path.name:
            raise ValueError("LoRA filename ไม่ตรงกับ local file ที่เลือก")
        return str(path.parent), path.name
    if source.startswith(("/", "./", "../", "~/")) and not path.is_dir():
        raise FileNotFoundError("ไม่พบ local LoRA path ที่เลือก")
    if weight_name:
        safe_relative_filename(weight_name, "LoRA filename")
        if not weight_name.endswith(".safetensors"):
            raise ValueError("LoRA filename ต้องลงท้าย .safetensors ตัวพิมพ์เล็ก")
        if path.is_dir() and not (path / weight_name).is_file():
            raise FileNotFoundError("ไม่พบ LoRA filename ภายใน local directory")
        return str(path) if path.is_dir() else source, weight_name
    if path.is_dir():
        candidates = [file.name for file in path.iterdir() if file.is_file() and file.name.endswith(".safetensors")]
    else:
        # Diffusers 0.36's implicit filename guess calls model_info without the
        # explicit token. Resolve here to support Colab Secrets / private repos.
        from huggingface_hub import list_repo_files
        candidates = [name for name in list_repo_files(repo_id=source, token=HF_TOKEN)
                      if name.endswith(".safetensors")]
    if not candidates:
        raise ValueError("ไม่พบไฟล์ .safetensors สำหรับ LoRA; ระบุ filename ที่ถูกต้อง (local directory ค้นหาเฉพาะชั้นแรก)")
    if len(candidates) != 1:
        raise ValueError("มีไฟล์ .safetensors มากกว่า 1 ไฟล์; ระบุ LoRA filename เอง ไม่เลือกให้โดยเดา")
    filename = safe_relative_filename(candidates[0], "LoRA filename")
    return str(path) if path.is_dir() else source, filename


def configure_vae_tiling(pipe, enabled):
    if not enabled:
        return "off"
    # Modern pipelines need not expose the deprecated pipeline-level wrapper.
    vae = getattr(pipe, "vae", None)
    modern = getattr(vae, "enable_tiling", None)
    legacy = getattr(pipe, "enable_vae_tiling", None)
    if callable(modern):
        modern()
    elif callable(legacy):
        legacy()
    else:
        print("Pipeline/VAE ไม่มี tiling API; ข้าม (ไม่ได้ลด transformer memory)")
        return "unavailable"
    return "enabled"


def get_pipeline(settings):
    global PIPE, PIPE_KEY, PIPE_LOAD_INFO
    mode, dtype = select_runtime(settings)
    # Include all loader-affecting settings, not prompt/seed. Authentication is never serialized.
    fields = ("source", "source_type", "pipeline", "revision", "single_file_config", "backend",
              "vae_tiling", "lora_source", "lora_weight_name", "lora_scale")
    loader = load_diffusers if settings["backend"] == "diffusers" else CUSTOM_LOADERS.get(settings["backend"])
    if loader is None:
        raise ValueError("Backend ยังไม่ลงทะเบียนใน CUSTOM_LOADERS: " + settings["backend"])
    # Custom loaders can consume ANY setting. Reusing only the Diffusers subset
    # could silently return a model loaded with an outdated custom configuration.
    loader_settings = {k:settings[k] for k in fields} if settings["backend"] == "diffusers" else settings
    key = (json.dumps(loader_settings, sort_keys=True), mode, str(dtype), id(loader))
    if PIPE is not None and key == PIPE_KEY:
        return PIPE, mode, str(dtype)
    lora_source, lora_weight_name = resolve_lora_source(settings)
    unload_model()
    try:
        PIPE = loader(settings, dtype)
        if settings["lora_source"]:
            if not all(hasattr(PIPE, name) for name in ("load_lora_weights", "set_adapters")):
                raise ValueError("Pipeline นี้ไม่มี LoRA adapter API")
            opts = dict(adapter_name="user_lora", token=HF_TOKEN, use_safetensors=True,
                        weight_name=lora_weight_name)
            PIPE.load_lora_weights(lora_source, **opts)
            PIPE.set_adapters(["user_lora"], adapter_weights=[settings["lora_scale"]])
        tiling_status = configure_vae_tiling(PIPE, settings["vae_tiling"])
        if mode in ("GPU", "CPU"):
            PIPE.to("cuda" if mode == "GPU" else "cpu")
        else:
            method = "enable_model_cpu_offload" if mode == "Model CPU offload" else "enable_sequential_cpu_offload"
            if not hasattr(PIPE, method):
                raise ValueError(f"Pipeline ไม่รองรับ {mode}; เลือก GPU/CPU หรือใช้ adapter")
            getattr(PIPE, method)()
        PIPE_LOAD_INFO = dict(lora_weight_name=lora_weight_name, vae_tiling=tiling_status)
        PIPE_KEY = key
        return PIPE, mode, str(dtype)
    except (Exception, KeyboardInterrupt):
        try:
            unload_model()
        except Exception:
            print("คืนหน่วยความจำหลังโหลดล้มเหลวไม่สำเร็จ; อาจต้อง restart runtime")
        raise

### เพิ่ม backend สำหรับโมเดลนอก Diffusers (ขั้นสูง)

ไม่มี generic loader ที่อ่านได้ทุกสถาปัตยกรรม: ต้องติดตั้ง dependencies ตาม official repository และเขียน wrapper ให้ตรง API ก่อน ไม่ดาวน์โหลด/รันโค้ดจาก repo ที่ไม่รู้จักอัตโนมัติ

เพิ่ม code cell **ก่อนเซลล์ Generate** แล้วลงทะเบียน:
```python
# def load_my_backend(settings, dtype):
#     # Load real weights with the model's official implementation.
#     # Return your wrapper; do not return a placeholder image.
#     return MyTextToImageWrapper(...)
# CUSTOM_LOADERS["my_backend"] = load_my_backend
```
จากนั้นเลือก `MODEL = "Custom"`, ใส่ source และ `BACKEND = "my_backend"`.
Wrapper ต้องมี `.to(device)` และ `__call__(prompt, width, height, num_inference_steps, generator, ...)` แบบ explicit signature; คืน object ที่มี `.images` เป็น list ของ PIL images. ถ้ามี safety flags ให้คืน `.nsfw_content_detected` / `.unsafe_images` ด้วย กรณีไม่มี offload API ให้เลือก `GPU` หรือ `CPU`. แปลง seed/settings ไปยัง API จริงของโมเดลภายใน wrapper; **การลงทะเบียนชื่ออย่างเดียวไม่ได้ทำให้โมเดลรองรับ**.

สำหรับ Custom Diffusers repo ปกติ ไม่ต้องเขียน adapter — ใช้ `BACKEND = "diffusers"`, `PIPELINE_CLASS = "Auto"` ได้เลย.

In [ ]:
#@title 7 — Generation + export engine
from datetime import datetime, timezone
import hashlib
import zipfile
import os


def write_metadata(run_dir, metadata):
    path = run_dir / "metadata.json"
    temporary = run_dir / "metadata.json.part"
    try:
        if temporary.is_symlink() or path.is_symlink():
            raise ValueError("Metadata path must not be a symlink")
        payload = json.dumps(redact_metadata(metadata), ensure_ascii=False, indent=2, allow_nan=False)
        temporary.write_text(payload, encoding="utf-8")
        temporary.replace(path)
    finally:
        try:
            temporary.unlink(missing_ok=True)
        except OSError:
            print("ไม่สามารถลบ metadata ชั่วคราวได้; ตรวจพื้นที่และสิทธิ์ filesystem")


def generate_images(settings, progress_callback=None):
    # Reset links first: failed runs must not offer an old archive as a new result.
    global LAST_RUN_DIR, LAST_ZIP
    LAST_RUN_DIR = LAST_ZIP = None
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + secrets.token_hex(3)
    run_dir = WORK_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=False, mode=0o700)
    pipe = result = None
    archive = WORK_DIR / (run_id + ".zip")
    temporary_archive = archive.with_suffix(".zip.part")
    complete = False
    metadata = None
    try:
        if progress_callback:
            progress_callback("loading", 0, settings["count"])
        pipe, mode, dtype = get_pipeline(settings)
        metadata = dict(settings=settings, runtime=dict(mode=mode, dtype=dtype,
            pipeline_class=type(pipe).__name__, python=platform.python_version(),
            loader=dict(PIPE_LOAD_INFO),
            packages={name:version(name) for name in ("torch", "diffusers", "transformers", "accelerate")}),
            created_at=datetime.now(timezone.utc).isoformat(), images=[], warnings=[], status="running")
        for index in range(settings["count"]):
            if progress_callback:
                progress_callback("generating", index, settings["count"])
            image_seed = settings["seed"] + index
            # CPU generator is supported by Diffusers randn_tensor and CPU offload.
            generator = torch.Generator(device="cpu").manual_seed(image_seed)
            kwargs, notes = build_call_kwargs(pipe, settings, generator)
            metadata["warnings"] = sorted(set(metadata["warnings"] + notes))
            for note in notes:
                print("⚠", note)
            print(f"Generating {index+1}/{settings['count']} | seed={image_seed}")
            with torch.inference_mode():
                result = pipe(**kwargs)
            images = getattr(result, "images", None)
            if images is None or len(images) != 1 or not isinstance(images[0], Image.Image):
                raise RuntimeError("Pipeline ต้องคืน .images เป็น list ของ PIL image จำนวน 1 ภาพ; ใช้ adapter สำหรับ API อื่น")
            for flag in ("nsfw_content_detected", "unsafe_images"):
                flags = getattr(result, flag, None)
                if flags is not None and any(flags):
                    raise RuntimeError("Pipeline safety checker flagged the output; ไม่ export ภาพนี้")
            image = images[0]
            filename = f"image_{index+1:03d}_seed_{image_seed}.png"
            image.save(run_dir / filename)
            record = dict(file=filename, seed=image_seed, width=image.width, height=image.height,
                          sha256=hashlib.sha256((run_dir / filename).read_bytes()).hexdigest())
            if settings["export_jpeg"]:
                # Flatten alpha onto white; JPEG cannot preserve transparency.
                rgba = image.convert("RGBA")
                rgb = Image.new("RGB", rgba.size, "white")
                rgb.paste(rgba, mask=rgba.getchannel("A"))
                jpg_name = filename.removesuffix(".png") + ".jpg"
                rgb.save(run_dir / jpg_name, quality=95, subsampling=0)
                record["jpeg"] = jpg_name
            metadata["images"].append(record)
            # Preserve completed-image metadata if a later image fails.
            write_metadata(run_dir, metadata)
            display(image)
            result = None
        if progress_callback:
            progress_callback("saving", settings["count"], settings["count"])
        metadata["status"] = "complete"
        write_metadata(run_dir, metadata)
        (run_dir / "README.txt").write_text(
            "AI-generated images. Check the source model and LoRA licenses before use.\n"
            "metadata.json contains prompts/settings; known tokens are redacted. Review before sharing.\n"
            "Seeds improve repeatability, not bitwise determinism across hardware/library versions.\n",
            encoding="utf-8")
        # Publish only a closed, complete archive. Interrupted writes must not leave
        # a plausible-looking .zip that the user could mistake for a successful run.
        # ZIP lives beside (not inside) the private run directory. Set 0600 at
        # creation so prompts/metadata are never written into a world-readable ZIP.
        fd = os.open(temporary_archive, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
        with os.fdopen(fd, "wb") as archive_stream, zipfile.ZipFile(archive_stream, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
            # Do not zip arbitrary files left by a callback / tool in this directory.
            names = ["metadata.json", "README.txt"]
            for record in metadata["images"]:
                names.append(record["file"])
                if "jpeg" in record:
                    names.append(record["jpeg"])
            for name in sorted(names):
                file = run_dir / name
                if file.is_symlink() or not file.is_file() or file.resolve().parent != run_dir.resolve():
                    raise ValueError("Export requires regular files within this run directory")
                bundle.write(file, arcname=name)
        temporary_archive.replace(archive)
        LAST_RUN_DIR, LAST_ZIP = run_dir, archive
        complete = True
        print("Saved:", run_dir, "\nZIP:", archive)
        return archive
    except (Exception, KeyboardInterrupt):
        # Avoid serializing exceptions (which could contain private paths/URLs).
        # Use in-memory metadata: a failed write may leave a truncated disk file.
        # Error reporting must not replace the original exception (e.g. disk full).
        if metadata is not None:
            metadata["status"] = "failed_partial" if metadata["images"] else "failed"
            try:
                write_metadata(run_dir, metadata)
            except Exception:
                print("ไม่สามารถบันทึกสถานะงานที่ล้มเหลวได้; ตรวจพื้นที่และสิทธิ์ filesystem")
        print("Run ไม่สำเร็จ; ไม่มี ZIP ใหม่ ภาพที่เสร็จแล้ว (ถ้ามี) อยู่ที่", run_dir)
        raise
    finally:
        pipe = result = None
        if not settings["keep_model"] or not complete:
            try:
                unload_model()
            except Exception:
                print("คืนหน่วยความจำไม่สำเร็จ; อาจต้อง restart runtime")
        if not complete:
            # Best effort: filesystem errors must not mask the original failure.
            for partial_archive in (temporary_archive, archive):
                try:
                    partial_archive.unlink(missing_ok=True)
                except OSError:
                    print("ลบ archive ที่ไม่สมบูรณ์ไม่ได้:", partial_archive)

In [ ]:
#@title ⚙️ Legacy generate (optional) — ผู้ใช้ทั่วไปไม่ต้องแก้ ให้ใช้แผงด้านล่าง
RUN_GENERATE = False
LAST_RUN_DIR = LAST_ZIP = None
if RUN_GENERATE:
    try:
        settings = resolve_settings(globals())
        print("Model:", settings["source"], "| Pipeline:", settings["pipeline"])
        print("Size:", settings["width"], "x", settings["height"], "| Base seed:", settings["seed"])
        generate_images(settings)
    except torch.cuda.OutOfMemoryError:
        unload_model()
        print("GPU OOM: ลด resolution / เลือก Sequential CPU offload / ใช้โมเดลเล็กลง แล้วรันใหม่")
        print("CPU offload ยังต้องใช้ system RAM; ไม่สามารถทำให้ทุกโมเดลพอดีกับ Colab ฟรีได้")
        raise
    except Exception:
        print("ตรวจ traceback: 401/403 → สิทธิ์ gated repo + HF_TOKEN; 404 → model ID/revision")
        print("Class/import error → ติดตั้ง dependencies แล้ว restart; unsupported architecture → ใช้ local adapter")
        print("LoRA mismatch → ตรวจ base architecture; missing safetensors → ใช้ checkpoint ที่ปลอดภัยและรองรับ")
        raise
else:
    print("ระบบพร้อม: ใช้ปุ่มสร้างภาพในแผงด้านล่าง (legacy RUN_GENERATE ยังปิดอยู่)")


In [ ]:
#@title ⚙️ Legacy download (optional) — ผู้ใช้ทั่วไปไม่ต้องแก้ ให้ใช้แผงด้านล่าง
DOWNLOAD_ZIP = True
if LAST_ZIP is None or not LAST_ZIP.exists():
    print("ยังไม่มี ZIP จากการ generate ที่สำเร็จในรอบนี้")
elif DOWNLOAD_ZIP:
    try:
        from google.colab import files
    except ImportError:
        # FileLink works when output is beneath the Jupyter notebook directory.
        display(FileLink(os.path.relpath(LAST_ZIP, Path.cwd())))
    else:
        files.download(str(LAST_ZIP))
else:
    print("ZIP:", LAST_ZIP)


In [ ]:
#@title ⚙️ Legacy cleanup (optional) — ผู้ใช้ทั่วไปไม่ต้องแก้ ให้ใช้แผงด้านล่าง
UNLOAD_NOW = False
if UNLOAD_NOW:
    unload_model()
    print("Released model references; HF disk cache retained")


## Tips / troubleshooting

- **LoRA file selection:** หากไม่ระบุ filename ระบบใช้ token ในการค้นหา repo และเลือกให้อัตโนมัติเฉพาะเมื่อมี `.safetensors` เพียงไฟล์เดียว หากมีหลายไฟล์ให้ระบุ filename ในขั้นสูง (รองรับ nested filename) สำหรับ local directory ค้นหาอัตโนมัติเฉพาะชั้นแรก
- **Checkpoint suffix:** ใช้ `.safetensors` ตัวพิมพ์เล็กเท่านั้น เพื่อให้ตรงกับการเลือกตัวอ่านไฟล์ของ dependency ไม่เปลี่ยนไปใช้ตัวอ่าน serialization อื่นโดยไม่ตั้งใจ
- **VAE tiling:** รองรับทั้ง `pipe.vae.enable_tiling()` และ wrapper แบบเดิม บันทึกสถานะที่ใช้จริงและ LoRA filename ที่เลือกไว้ใน `metadata.json → runtime → loader`


- **ขอบเขตความปลอดภัย:** เป็น Notebook ส่วนตัว ไม่ใช่ multi-user service หรือ sandbox สำหรับโค้ดที่ไม่ไว้ใจ ใช้ runtime ที่เชื่อถือได้ และอย่าเปิด public endpoint ให้คนอื่นส่ง settings
- **Secrets:** ใส่ token เฉพาะ Colab Secrets / environment ไม่ใส่ใน prompt, LoRA URL หรือ Extra kwargs; ระบบปิดบัง token ที่รู้จักใน metadata และ error ของแผง แต่ไม่สามารถรับรอง log จาก dependency, traceback ของ legacy cells หรือข้อความที่ถูกสร้างลงในภาพได้
- **ไฟล์ภายนอก:** Single-file URL รองรับ Hugging Face เท่านั้น; filename ใน URL/LoRA ต้องไม่เป็น absolute path หรือมี `..` ส่วน local model paths ยังเป็นการเข้าถึงไฟล์ที่ผู้ใช้เลือกเองตามปกติ ไม่ใช่ filesystem sandbox
- **ZIP:** รวมเฉพาะภาพที่สร้าง, metadata และ README; ไม่รวมไฟล์อื่นในโฟลเดอร์และไม่ตาม symlink ของไฟล์ที่จะ export
- **Extra kwargs:** จำกัด 32 KiB / ความลึก 12 ชั้น เพื่อลดการค้างจาก JSON ผิดรูปแบบ ไม่ได้ทำให้ทุก parameter ปลอดภัยต่อ RAM/VRAM โดยอัตโนมัติ


- **แผงควบคุม:** Run all ครั้งแรก แล้วใช้แผงด้านล่างได้เลย ไม่ต้องแก้ Internal defaults หรือ Legacy cells
- **สัดส่วน:** คงอัตราส่วนที่เลือกและจัด pixel ให้หารตามเงื่อนไขโมเดลลงตัว ระดับ 512/768/1024/1536 อิงพื้นที่ใกล้เคียงด้าน² ไม่ใช่ความยาวด้านใดด้านหนึ่ง ขนาดบนแผงคือขนาดที่ร้องขอ ส่วน metadata เก็บขนาดภาพที่ pipeline คืนจริง
- **ค่าแนะนำ:** เปลี่ยนโมเดลจะคืน Steps/Guidance, resolution Auto, source/pipeline options และล้าง LoRA/Extra kwargs; Prompt / สัดส่วน / จำนวนภาพ / seed จะไม่ถูกล้าง
- **ดาวน์โหลด:** ปุ่ม “ดาวน์โหลด ZIP ล่าสุด” อ้างถึงงานสำเร็จจากแผงนี้ ไม่ใช่ ZIP จากเซลล์หรือแผงเก่า; เมื่อเริ่มงานใหม่ที่ล้มเหลวจะไม่มีลิงก์เก่ามาปะปน


- **Custom model:** ใช้ repo ที่มี `model_index.json` และ safetensors; ถ้าเป็น fine-tune ของ SDXL ให้ใช้ค่า SDXL เป็นแนวทาง (1024px / 30 steps / guidance 5) ไม่ใช้ค่าของ Turbo โดยอัตโนมัติ
- **Single-file URL:** รองรับเฉพาะ Hugging Face `https://huggingface.co/org/repo/resolve/revision/file.safetensors` (หรือ `blob`) ไม่มี query/token ใน URL; revision ใน URL จะถูกใช้จริง แหล่งอื่นให้ดาวน์โหลดลง Colab แล้วใส่ local path ก่อน ส่วน `SINGLE_FILE_CONFIG` ใช้ revision เริ่มต้นของ config repo — หากต้องการ pin config ให้ดาวน์โหลด config เป็น local directory
- **Qwen Image guidance:** ช่อง `GUIDANCE` ควบคุม `true_cfg_scale` โดยตรง (default 4) ไม่ส่ง `guidance_scale` ที่ base model ไม่ได้ใช้; ห้ามตั้งค่าเดียวกันซ้ำใน Extra kwargs
- **Single-file:** ตั้ง Custom + Single safetensors file + `StableDiffusionPipeline` หรือ `StableDiffusionXLPipeline` ตาม checkpoint; กรณีตรวจ config ไม่ได้ ระบุ `SINGLE_FILE_CONFIG` เป็น Diffusers config repo ที่ตรงกัน ต้องมีอินเทอร์เน็ตสำหรับ tokenizer/config ที่ยังไม่อยู่ใน cache
- **LoRA:** ต้องเป็นสถาปัตยกรรมเดียวกับ base model และอ่าน license ทั้งสองตัว; รองรับหนึ่ง adapter ต่อรอบ ไม่โหลด pickle weights
- **Negative prompt:** ส่งเฉพาะ pipeline ที่ประกาศ parameter นี้; หากไม่รองรับจะแจ้งและจดไว้ใน metadata บางโมเดลยังต้องตั้ง CFG ตามคู่มือจึงจะมีผล
- **FP16 / BF16:** T4 ไม่รองรับ BF16; preset ที่ต้องการ BF16 จะใช้ FP32 ใน Auto บน T4 ซึ่งอาจกิน RAM มาก แนะนำเริ่ม SD 1.5 / SDXL แทนโมเดลใหญ่มาก
- **Offload:** Model CPU offload เร็วกว่า sequential แต่ต้องพอสำหรับ component ใหญ่ที่สุด ส่วน sequential ช้ากว่าและยังใช้ RAM มาก; VAE tiling ลดเฉพาะส่วน VAE ไม่มีการแอบเปลี่ยนโมเดล/ขนาดภาพเมื่อ OOM
- **Disk:** weights อาจกินพื้นที่หลายสิบ GB เก็บใน Hugging Face cache ไม่อยู่ใน ZIP หรือ Git; Colab runtime reset จะลบไฟล์ชั่วคราว ให้ดาวน์โหลดก่อน
- **Reproducibility:** ตั้ง revision เป็น commit SHA และเก็บ metadata; seed เดิมไม่รับประกันภาพเหมือนกันข้าม GPU / library versions / mutable LoRA repo
- **Safety/license:** ไม่มีการรับรองสิทธิ์เชิงพาณิชย์หรือการผ่าน stock platform; อย่าใช้ชื่อ preset แทนการอ่าน license จริง

### Verification scope
Notebook includes offline tests for validation, argument routing, loader configuration and export orchestration. A successful syntax/unit test does **not** verify model downloads or real GPU inference; each model must be smoke-tested on a suitable runtime with its actual access permissions.

In [ ]:
#@title ⚙️ UI helpers — คำนวณสัดส่วนและเชื่อมกับ generation engine
ASPECT_RATIOS = {"1:1": (1, 1), "3:2": (3, 2), "2:3": (2, 3), "4:3": (4, 3),
                 "3:4": (3, 4), "16:9": (16, 9), "9:16": (9, 16)}
RESOLUTION_OPTIONS = [("อัตโนมัติตามโมเดล", "Auto"), ("512 · ประหยัดหน่วยความจำ", "512"),
                      ("768 · ปานกลาง", "768"), ("1024 · รายละเอียดสูง", "1024"),
                      ("1536 · ใช้หน่วยความจำมาก", "1536")]


def ui_dimensions(model, ratio, resolution="Auto", width=512, height=512):
    if model != "Custom" and model not in MODEL_REGISTRY:
        raise ValueError("Unknown model: " + model)
    preset = MODEL_REGISTRY.get(model, {})
    multiple = preset.get("multiple", 32)
    if ratio == "Custom":
        if any(type(n) is not int or not 128 <= n <= 4096 or n % multiple for n in (width, height)):
            raise ValueError(f"ขนาดต้องอยู่ในช่วง 128–4096 และหาร {multiple} ลงตัว")
        return width, height
    if ratio not in ASPECT_RATIOS:
        raise ValueError("Unknown aspect ratio: " + ratio)
    if resolution not in dict((value, label) for label, value in RESOLUTION_OPTIONS):
        raise ValueError("Unknown resolution: " + str(resolution))
    base = preset.get("size", 1024) if resolution == "Auto" else int(resolution)
    a, b = ASPECT_RATIOS[ratio]
    # Keep the requested ratio EXACT and dimensions divisible by model alignment.
    # Pixel area is approximately base², not base pixels on the long edge.
    unit = multiple * max(1, round(base / (multiple * math.sqrt(a * b))))
    return unit * a, unit * b


def ui_settings(values, ratio, resolution):
    values = dict(values)  # Never mutate legacy globals or a caller's config.
    values["WIDTH"], values["HEIGHT"] = ui_dimensions(
        values["MODEL"], ratio, resolution, values["WIDTH"], values["HEIGHT"])
    settings = resolve_settings(values)
    settings["ui"] = dict(aspect_ratio=ratio, resolution=resolution)
    return settings


def download_archive(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError("ไม่พบ ZIP แล้ว กรุณาสร้างภาพใหม่หรือดูโฟลเดอร์ output")
    try:
        from google.colab import files
    except ImportError:
        display(FileLink(os.path.relpath(path, Path.cwd())))
    else:
        files.download(str(path))


In [ ]:
#@title 🎨 Image Generator — เลือกโมเดล → Prompt → สัดส่วน → สร้างภาพ
import html
import ipywidgets as widgets
from IPython.display import clear_output

# Required by Colab for widget communication; no external server or public tunnel.
try:
    from google.colab import output as colab_output
except ImportError:
    pass
else:
    colab_output.enable_custom_widget_manager()


class GeneratorDashboard:
    def __init__(self, defaults):
        self.defaults = {name: defaults[name] for name in SETTING_FIELDS}
        self.busy = False
        self.closed = False
        self.updating = False
        self.last_archive = None
        self.last_settings = None
        self.observers = []
        full = lambda: widgets.Layout(width="100%")
        self.controls = c = {
            "MODEL": widgets.Dropdown(options=[*MODEL_REGISTRY, "Custom"], value="SD 1.5", layout=full()),
            "PROMPT": widgets.Textarea(value=defaults["PROMPT"], placeholder="Describe your image: subject, scene, lighting, style…", layout=widgets.Layout(width="100%", height="130px")),
            "NEGATIVE_PROMPT": widgets.Textarea(value="", placeholder="สิ่งที่ไม่ต้องการในภาพ เช่น blurry, low quality (บางโมเดลไม่ใช้ค่านี้)", layout=widgets.Layout(width="100%", height="76px")),
            "RATIO": widgets.ToggleButtons(options=[(ratio, ratio) for ratio in ASPECT_RATIOS] + [("กำหนดเอง", "Custom")], value="1:1", style={"button_width":"85px"}),
            "RESOLUTION": widgets.Dropdown(options=RESOLUTION_OPTIONS, value="Auto", layout=full()),
            "WIDTH": widgets.BoundedIntText(value=512, min=128, max=4096, step=8, description="Width", layout=full()),
            "HEIGHT": widgets.BoundedIntText(value=512, min=128, max=4096, step=8, description="Height", layout=full()),
            "STEPS": widgets.IntSlider(value=30, min=1, max=200, continuous_update=False, layout=full()),
            "GUIDANCE": widgets.FloatSlider(value=7.5, min=0, max=30, step=0.1, readout_format=".1f", continuous_update=False, layout=full()),
            "NUM_IMAGES": widgets.IntSlider(value=1, min=1, max=8, continuous_update=False, layout=full()),
            "RANDOM_SEED": widgets.Checkbox(value=True, description="สุ่ม seed ทุกครั้ง", indent=False),
            "SEED": widgets.IntText(value=42, description="Seed", layout=full()),
            "PRECISION": widgets.Dropdown(options=["Auto", "FP16", "BF16", "FP32"], value="Auto", layout=full()),
            "MEMORY_MODE": widgets.Dropdown(options=["Auto", "GPU", "Model CPU offload", "Sequential CPU offload", "CPU"], value="Auto", layout=full()),
            "VAE_TILING": widgets.Checkbox(value=True, description="VAE tiling · ลดหน่วยความจำส่วน VAE", indent=False),
            "EXPORT_JPEG": widgets.Checkbox(value=True, description="บันทึก JPEG เพิ่มจาก PNG", indent=False),
            "KEEP_MODEL_IN_MEMORY": widgets.Checkbox(value=False, description="เก็บโมเดลใน RAM/VRAM เพื่อสร้างรอบต่อไปเร็วขึ้น", indent=False),
            "CUSTOM_MODEL": widgets.Text(placeholder="org/model หรือ /content/model.safetensors", layout=full()),
            "SOURCE": widgets.Dropdown(options=["Repository / directory", "Single safetensors file"], layout=full()),
            "PIPELINE_CLASS": widgets.Text(value="Auto", placeholder="เช่น StableDiffusionXLPipeline", layout=full()),
            "SINGLE_FILE_CONFIG": widgets.Text(placeholder="Optional matching Diffusers config repo", layout=full()),
            "BACKEND": widgets.Text(value="diffusers", layout=full()),
            "REVISION": widgets.Text(placeholder="เว้นว่าง = default branch; หรือระบุ commit SHA", layout=full()),
            "LORA_SOURCE": widgets.Text(placeholder="Optional: org/lora หรือ local path", layout=full()),
            "LORA_WEIGHT_NAME": widgets.Text(placeholder="Optional: adapter.safetensors", layout=full()),
            "LORA_SCALE": widgets.FloatSlider(value=1, min=0, max=2, step=0.05, continuous_update=False, layout=full()),
            "EXTRA_KWARGS_JSON": widgets.Textarea(value="{}", layout=widgets.Layout(width="100%", height="70px")),
        }
        def field(label, control, hint=""):
            children = [widgets.HTML(f"<b>{html.escape(label)}</b>"), control]
            if hint:
                children.append(widgets.HTML(f"<small>{html.escape(hint)}</small>"))
            return widgets.VBox(children, layout=widgets.Layout(flex="1 1 240px", min_width="220px"))
        def row(*children):
            return widgets.Box(children, layout=widgets.Layout(display="flex", flex_flow="row wrap", grid_gap="14px", width="100%"))
        self.model_info = widgets.HTML()
        self.dimension_preview = widgets.HTML()
        self.summary = widgets.HTML()
        self.custom_box = widgets.VBox([
            field("Hugging Face ID / local path", c["CUSTOM_MODEL"]),
            row(field("รูปแบบโมเดล", c["SOURCE"]), field("Pipeline class", c["PIPELINE_CLASS"], "Single-file ต้องระบุ class ให้ตรง เช่น StableDiffusionXLPipeline")),
            row(field("Single-file config (ถ้ามี)", c["SINGLE_FILE_CONFIG"]), field("Backend", c["BACKEND"], "ค่าอื่นนอกจาก diffusers ต้องลงทะเบียน local adapter ก่อน")),
        ], layout=widgets.Layout(border="1px solid #cedbd6", padding="12px", margin="8px 0"))
        self.guidance_label = widgets.HTML()
        self.advanced = widgets.Accordion(children=[
            widgets.VBox([field("Negative prompt", c["NEGATIVE_PROMPT"]), c["RANDOM_SEED"], c["SEED"],
                          widgets.HTML("<small>ปิดการสุ่มเพื่อใช้ seed เดิม แต่ภาพอาจต่างกันข้าม GPU / library versions</small>")]),
            widgets.VBox([row(field("Precision", c["PRECISION"]), field("Memory mode", c["MEMORY_MODE"])),
                          c["VAE_TILING"], c["EXPORT_JPEG"], c["KEEP_MODEL_IN_MEMORY"], field("Model revision", c["REVISION"])]),
            widgets.VBox([field("LoRA source", c["LORA_SOURCE"]), field("LoRA filename", c["LORA_WEIGHT_NAME"]),
                          field("LoRA strength", c["LORA_SCALE"]), field("Extra kwargs (JSON)", c["EXTRA_KWARGS_JSON"], "ไม่จำเป็นสำหรับการใช้งานทั่วไป; ต้องเป็น parameter ที่ pipeline รองรับ")]),
        ], selected_index=None, layout=full())
        for i, title in enumerate(["ขั้นสูง · Negative prompt & Seed", "ขั้นสูง · หน่วยความจำ & ไฟล์", "ขั้นสูง · LoRA & Extra parameters"]):
            self.advanced.set_title(i, title)
        self.generate_button = widgets.Button(description="สร้างภาพ", icon="magic", button_style="success", layout=widgets.Layout(width="170px", height="42px"))
        self.download_button = widgets.Button(description="ดาวน์โหลด ZIP ล่าสุด", icon="download", button_style="info", disabled=True, layout=widgets.Layout(width="210px", height="42px"))
        self.reset_button = widgets.Button(description="คืนค่าแนะนำของโมเดล", icon="refresh", layout=widgets.Layout(width="200px"))
        self.unload_button = widgets.Button(description="คืน RAM / VRAM", icon="eraser", layout=widgets.Layout(width="170px"))
        self.status = widgets.HTML()
        self.progress = widgets.IntProgress(value=0, min=0, max=1, description="ภาพที่เสร็จ", layout=full())
        self.output = widgets.Output(layout=widgets.Layout(width="100%"))
        self.root = widgets.VBox([
            widgets.HTML('<div style="padding:22px 24px;border-radius:12px;background:#142b25;color:#f2faf6">'
                         '<small style="color:#9cd9bc;letter-spacing:2px">OPEN-MODEL WORKBENCH</small>'
                         '<h2 style="margin:8px 0;color:#f2faf6">สร้างภาพใน 4 ขั้นตอน</h2>'
                         '<div>เลือกโมเดล → เขียน Prompt → เลือกสัดส่วน → กดสร้างภาพ</div></div>'),
            field("01 · เลือกโมเดล", c["MODEL"], "SD 1.5 เหมาะสำหรับเริ่มต้น; การเปลี่ยนโมเดลจะคืนค่า Steps / Guidance และล้าง LoRA / Extra kwargs"),
            self.model_info, self.custom_box,
            field("02 · ภาพที่อยากได้ (Prompt)", c["PROMPT"], "ภาษาอังกฤษมักใช้ได้กว้างกว่า: ระบุสิ่งที่ต้องการ ฉาก แสง และสไตล์"),
            field("03 · สัดส่วนภาพ", c["RATIO"]),
            row(field("ระดับความละเอียด", c["RESOLUTION"], "อิงพื้นที่ภาพใกล้เคียงด้าน²; แสดงขนาดจริงด้านล่าง"), self.dimension_preview),
            row(c["WIDTH"], c["HEIGHT"]),
            widgets.HTML("<b>04 · ปรับค่าการสร้าง (หรือใช้ค่าที่แนะนำไว้แล้ว)</b>"),
            row(field("Steps", c["STEPS"], "จำนวนรอบประมวลผล · มากขึ้นใช้เวลามากขึ้น ไม่รับประกันคุณภาพที่ดีกว่า"),
                widgets.VBox([self.guidance_label, c["GUIDANCE"], widgets.HTML("<small>ความแรงในการทำตาม prompt · ค่าแนะนำต่างกันตามโมเดล</small>")], layout=widgets.Layout(flex="1 1 240px", min_width="220px")),
                field("จำนวนภาพ", c["NUM_IMAGES"], "สร้างทีละภาพเพื่อลดการใช้ VRAM")),
            self.advanced, row(self.reset_button, self.unload_button), self.summary,
            row(self.generate_button, self.download_button), self.status, self.progress,
            widgets.HTML("<small>ครั้งแรกต้องดาวน์โหลด weights อาจใช้เวลานานและพื้นที่หลาย GB · หยุดงานด้วย Runtime → Interrupt execution / ปุ่ม Stop ของ Notebook</small>"),
            self.output,
        ], layout=widgets.Layout(width="100%", max_width="1040px", grid_gap="12px", padding="12px"))
        for key, control in c.items():
            callback = self._model_changed if key == "MODEL" else self._changed
            control.observe(callback, names="value")
            self.observers.append((control, callback))
        self.generate_button.on_click(self._generate)
        self.download_button.on_click(self._download)
        self.reset_button.on_click(self._reset)
        self.unload_button.on_click(self._unload)
        self._reset()
        self._set_status("พร้อมสร้างภาพ · ยังไม่โหลด weights", "#315e4c")

    def _set_status(self, message, color="#315e4c"):
        self.status.value = f'<div role="status" style="padding:10px;border-left:4px solid {color}">{html.escape(message)}</div>'

    def _model_changed(self, change):
        if not self.busy and not self.updating:
            self._reset()
            self._set_status("เปลี่ยนโมเดลแล้ว · ใช้ Steps/Guidance แนะนำและล้าง LoRA/Extra kwargs เดิม")

    def _changed(self, change):
        if not self.busy and not self.updating:
            self._refresh()

    def _reset(self, _=None):
        if self.busy:
            return
        self.updating = True
        c = self.controls
        preset = MODEL_REGISTRY.get(c["MODEL"].value, {})
        for name, value in {"STEPS":preset.get("steps", 30), "GUIDANCE":preset.get("guidance", 5.0),
                            "RESOLUTION":"Auto", "PIPELINE_CLASS":"Auto", "SOURCE":"Repository / directory",
                            "SINGLE_FILE_CONFIG":"", "BACKEND":"diffusers", "REVISION":"",
                            "LORA_SOURCE":"", "LORA_WEIGHT_NAME":"", "LORA_SCALE":1.0, "EXTRA_KWARGS_JSON":"{}"}.items():
            c[name].value = value
        # Prompt, aspect ratio, count, negative prompt and seed are deliberately preserved.
        self.updating = False
        self._refresh()

    def _refresh(self):
        c = self.controls
        self.updating = True
        try:
            custom_size = c["RATIO"].value == "Custom"
            c["WIDTH"].disabled = c["HEIGHT"].disabled = self.busy or not custom_size
            c["RESOLUTION"].disabled = self.busy or custom_size
            c["SEED"].disabled = self.busy or c["RANDOM_SEED"].value
            self.custom_box.layout.display = "" if c["MODEL"].value == "Custom" else "none"
            preset = MODEL_REGISTRY.get(c["MODEL"].value, {})
            note = preset.get("note", "Custom: ใช้โมเดลที่ Diffusers รองรับ หรือลงทะเบียน local adapter ก่อน")
            link = (f'<a href="https://huggingface.co/{html.escape(preset["repo"], quote=True)}" target="_blank" rel="noopener noreferrer">Model card / license ↗</a>' if preset else "")
            runtime = "CUDA พร้อมใช้" if torch.cuda.is_available() else "CPU เท่านั้น · แนะนำเปลี่ยนเป็น GPU runtime"
            self.model_info.value = f"<div>{html.escape(note)} · {link}<br><small>{runtime}</small></div>"
            self.guidance_label.value = "<b>Guidance · true CFG</b>" if c["MODEL"].value == "Qwen Image" else "<b>Guidance / CFG</b>"
            w, h = ui_dimensions(c["MODEL"].value, c["RATIO"].value, c["RESOLUTION"].value, c["WIDTH"].value, c["HEIGHT"].value)
            if not custom_size:
                c["WIDTH"].value, c["HEIGHT"].value = w, h
            thumb_w, thumb_h = round(100 * w / max(w, h)), round(100 * h / max(w, h))
            self.dimension_preview.value = (
                f'<div style="display:flex;align-items:center;gap:14px;padding:10px">'
                f'<div aria-label="Aspect ratio preview" style="width:{thumb_w}px;height:{thumb_h}px;border:2px solid #589a78;background:#e7f3ed;border-radius:5px"></div>'
                f'<div><b>{w} × {h} px</b><br><small>กรอบแสดงสัดส่วน ไม่ใช่ภาพที่สร้าง</small></div></div>')
            seed = "สุ่ม" if c["RANDOM_SEED"].value else str(c["SEED"].value)
            self.summary.value = f'<div><b>รอบถัดไป:</b> {html.escape(c["MODEL"].value)} · {w} × {h} · {c["STEPS"].value} steps · {c["NUM_IMAGES"].value} ภาพ · Seed {seed}</div>'
        except ValueError as exc:
            self.dimension_preview.value = ""
            self.summary.value = f'<div style="color:#a43131">{html.escape(str(exc))}</div>'
        finally:
            self.updating = False

    def snapshot(self):
        values = dict(self.defaults)
        values.update({name: control.value for name, control in self.controls.items() if name in self.defaults})
        if self.controls["RANDOM_SEED"].value:
            values["SEED"] = -1
        return ui_settings(values, self.controls["RATIO"].value, self.controls["RESOLUTION"].value)

    def _lock(self, busy):
        self.busy = busy
        for control in self.controls.values():
            control.disabled = busy
        for button in (self.generate_button, self.download_button, self.reset_button, self.unload_button):
            button.disabled = busy
        if not busy:
            self._refresh()
            self.download_button.disabled = self.last_archive is None

    def _progress(self, stage, completed, total):
        self.progress.max = total
        self.progress.value = completed
        text = {"loading":"กำลังโหลดโมเดล · ครั้งแรกอาจดาวน์โหลด weights หลาย GB",
                "generating":f"กำลังสร้างภาพ {min(completed + 1, total)}/{total}",
                "saving":"กำลังบันทึกภาพและจัดทำ ZIP"}
        self._set_status(text.get(stage, stage))

    def _generate(self, _=None):
        if self.busy or self.closed:
            return
        self.last_archive = self.last_settings = None
        self.progress.value = 0
        self.progress.bar_style = ""
        self._lock(True)
        try:
            with self.output:
                # Catch INSIDE Output: in a live IPython kernel __exit__ can
                # suppress exceptions. Catching outside can falsely report success.
                try:
                    clear_output(wait=True)
                    settings = self.snapshot()
                    self.last_settings = settings
                    self._set_status("เริ่มสร้างภาพ")
                    print(f"Model: {settings['model']} | Seed: {settings['seed']} | {settings['width']} × {settings['height']}")
                    archive = generate_images(settings, progress_callback=self._progress)
                    if archive is None or not Path(archive).is_file():
                        raise RuntimeError("Engine ไม่ได้คืนไฟล์ ZIP ที่มีอยู่จริง")
                    self.last_archive = Path(archive)
                    self.progress.value = self.progress.max
                    self.progress.bar_style = "success"
                    self._set_status(f"สำเร็จ {settings['count']} ภาพ · Seed เริ่มต้น {settings['seed']} · กดดาวน์โหลด ZIP ล่าสุดได้เลย")
                except KeyboardInterrupt:
                    self.last_archive = None
                    self.progress.bar_style = "warning"
                    self._set_status("หยุดการสร้างแล้ว · ไม่มี ZIP ใหม่", "#956522")
                except Exception as exc:
                    self.last_archive = None
                    self.progress.bar_style = "danger"
                    self._set_status("สร้างภาพไม่สำเร็จ · แก้ตามข้อความด้านล่างแล้วลองใหม่", "#a43131")
                    print(f"{type(exc).__name__}: {redact_text(exc)}")
                    if type(exc).__name__ == "OutOfMemoryError":
                        print("ลดความละเอียด / ใช้ Sequential CPU offload / เปลี่ยนโมเดลเล็กลง (offload ยังใช้ system RAM)")
                    else:
                        print("401/403: ตรวจสิทธิ์ gated model และ HF_TOKEN ใน Colab Secrets; import error: restart runtime หลังติดตั้ง")
        finally:
            self._lock(False)

    def _download(self, _=None):
        if self.busy or self.last_archive is None:
            return
        with self.output:
            try:
                download_archive(self.last_archive)
            except Exception as exc:
                print(f"ดาวน์โหลดไม่ได้: {redact_text(exc)}")
                self._set_status("ดาวน์โหลดไม่สำเร็จ · ดูข้อความในผลลัพธ์", "#a43131")

    def _unload(self, _=None):
        if not self.busy:
            unload_model()
            self._set_status("คืน RAM / VRAM แล้ว · ภาพและ ZIP ยังอยู่; weights ใน disk cache ไม่ถูกลบ")

    def close(self):
        if self.closed:
            return
        self.closed = True
        # Rerunning the UI cell must not leave duplicate live buttons/listeners.
        for control, callback in self.observers:
            control.unobserve(callback, names="value")
        for button, callback in ((self.generate_button, self._generate), (self.download_button, self._download),
                                 (self.reset_button, self._reset), (self.unload_button, self._unload)):
            button.on_click(callback, remove=True)
        def close_tree(widget):
            for child in getattr(widget, "children", ()):
                close_tree(child)
            widget.close()
        close_tree(self.root)


_old_ui = globals().get("GENERATOR_UI")
if _old_ui is not None:
    _old_ui.close()
GENERATOR_UI = GeneratorDashboard(DEFAULT_SETTINGS)
display(GENERATOR_UI.root)

del _old_ui
